In [4]:
import pandas as pd
import re
from collections import defaultdict

# =============================
# LOAD DATA
# =============================
file_path = "C:/Users/Ex0164/Tushar vats/Copy of Master Data _290102026 2.xlsx"

master = pd.read_excel(file_path, sheet_name="Master Data ")
ppm = pd.read_excel(file_path, sheet_name="Part Production Master ")

# =============================
# CLEAN NUMERIC COLUMNS
# =============================
for c in ["Daily Plan", "Sub Count", "Inventory_25", "Minimum Quantity"]:
    master[c] = pd.to_numeric(master[c], errors="coerce").fillna(0)

ppm["Machine"] = pd.to_numeric(ppm["Machine"], errors="coerce").fillna(0)

# =============================
# CYCLE TIME (SECONDS → MINUTES)
# =============================
cycle_time_min = {
    row["Material"]: row["Machine"] / 60
    for _, row in ppm.iterrows()
}

# =============================
# CONSTANTS
# =============================
ALLOWED_MACHINES = ["MP-01", "MP-05", "MP-10", "MP-17"]
MACHINE_CAPACITY = 22 * 60  # 1320 minutes

# =============================
# MACHINE NORMALIZER
# =============================
def normalize_machine(m):
    if not m or pd.isna(m):
        return None
    m = str(m).upper().strip()
    m = re.sub(r"[^A-Z0-9\-]", "", m)
    if m.startswith("MP") and "-" not in m:
        m = m.replace("MP", "MP-")
    return m

# =============================
# STEP 1: FILTER DAILY PLAN > 0
# =============================
valid = master[master["Daily Plan"] > 0].copy()

# =============================
# STEP 2: AGGREGATE (NO CLEVERNESS)
# =============================
records = []

for child, grp in valid.groupby("Child Part", sort=False):

    demand = (grp["Daily Plan"] * grp["Sub Count"]).sum()
    min_qty = grp["Minimum Quantity"].iloc[0]
    inv = grp["Inventory_25"].iloc[0]

    machines = ",".join(grp["Vertical Machines"].astype(str))

    records.append({
        "Child Part": child,
        "Required Qty": min_qty + demand,
        "Inventory_25": inv,
        "Vertical Machines": machines
    })

agg = pd.DataFrame(records)
agg["Net Required Qty"] = agg["Required Qty"] - agg["Inventory_25"]

# =============================
# TRACKING
# =============================
machine_load = {m: 0.0 for m in ALLOWED_MACHINES}
machine_plan = defaultdict(list)
rejections = []

# =============================
# STEP 3: ASSIGN AS-IS (NO BALANCING)
# =============================
for _, row in agg.iterrows():

    child = row["Child Part"]
    net_qty = row["Net Required Qty"]

    if net_qty <= 0:
        continue

    ct = cycle_time_min.get(child)
    if not ct or ct <= 0:
        rejections.append((child, "Missing cycle time"))
        continue

    remaining_time = net_qty * ct

    # Parse machines in GIVEN order
    raw = str(row["Vertical Machines"])
    tokens = re.split(r"[,\|/\\\n]+", raw)

    vertical = []
    for t in tokens:
        m = normalize_machine(t)
        if m and m in ALLOWED_MACHINES and m not in vertical:
            vertical.append(m)

    if not vertical:
        rejections.append((child, "No eligible machine"))
        continue

    # Assign sequentially
    for m in vertical:
        if remaining_time <= 0:
            break

        available = MACHINE_CAPACITY - machine_load[m]
        if available <= 0:
            continue

        alloc_time = min(available, remaining_time)
        alloc_qty = alloc_time / ct

        machine_load[m] += alloc_time
        remaining_time -= alloc_time

        machine_plan[m].append({
            "Child Part": child,
            "Quantity": round(alloc_qty, 2),
            "Time Used (min)": round(alloc_time, 2)
        })

    if remaining_time > 0:
        rejections.append((child, "Capacity shortfall"))

# =============================
# DISPLAY
# =============================
print("\n========== MACHINE-WISE PLAN ==========\n")
for m in ALLOWED_MACHINES:
    print(f"🔧 {m}")
    if machine_plan[m]:
        display(pd.DataFrame(machine_plan[m]))
    else:
        print("No allocation")
    print("-" * 60)

print("\n========== MACHINE LOAD SUMMARY ==========\n")
display(pd.DataFrame([
    {
        "Machine": m,
        "Used (min)": round(machine_load[m], 2),
        "Remaining (min)": round(MACHINE_CAPACITY - machine_load[m], 2)
    }
    for m in ALLOWED_MACHINES
]))

print("\n========== REJECTIONS ==========\n")
if rejections:
    display(pd.DataFrame(rejections, columns=["Child Part", "Reason"]))
else:
    print("✅ No rejections")


========== MACHINE-WISE PLAN ==========

🔧 MP-01


,Child Part,Quantity,Time Used (min)
0,S11434-002A0X,816.52,612.39
1,S12071-001A0X,41.35,24.12
2,S12094-003A0X,22.68,18.14
3,S41354-011A0X,887.13,665.35


------------------------------------------------------------
🔧 MP-05


,Child Part,Quantity,Time Used (min)
0,14SW030082-00001X0,689.71,517.28
1,14SW220197-00005X0,1926.52,802.72


------------------------------------------------------------
🔧 MP-10


...

,Child Part,Quantity,Time Used (min)
0,14SW110487-00009X0,285.26,114.10
1,14SW220201-00005X0,1152.10,691.26


------------------------------------------------------------

========== MACHINE LOAD SUMMARY ==========



,Machine,Used (min),Remaining (min)
0,MP-01,1320.00,0.00
1,MP-05,1320.00,0.00
2,MP-10,1031.65,288.35
3,MP-17,805.36,514.64



========== REJECTIONS ==========



,Child Part,Reason
0,14CL510026-00001X0,Missing cycle time
1,14CL510026-00001X1,No eligible machine
2,14CL510026-00002X0,No eligible machine
3,14CL510026-00003Y0,Missing cycle time
4,14CL510026-00004X0,No eligible machine
...,...,...
3794,W01007-001A0X,Missing cycle time
3795,W01007-001A1X,No eligible machine
3796,W01007-002A0Y,Missing cycle time
3797,W01008-000A0X,Missing cycle time


In [6]:
import pandas as pd
import re
from collections import defaultdict


file_path = "C:/Users/Ex0164/Tushar vats/Copy of Master Data _290102026 2.xlsx"

master = pd.read_excel(file_path, sheet_name="Master Data ")
ppm = pd.read_excel(file_path, sheet_name="Part Production Master ")


for c in ["Daily Plan", "Sub Count", "Inventory_25", "Minimum Quantity"]:
    master[c] = pd.to_numeric(master[c], errors="coerce").fillna(0)

ppm["Machine"] = pd.to_numeric(ppm["Machine"], errors="coerce").fillna(0)

# =============================
# CYCLE TIME (SECONDS → MINUTES)
# =============================
cycle_time_min = {
    row["Material"]: row["Machine"] / 60
    for _, row in ppm.iterrows()
}

# =============================
# CONSTANTS
# =============================
ALLOWED_MACHINES = ["MP-01", "MP-05", "MP-10", "MP-17"]
MACHINE_CAPACITY = 22 * 60  # 1320 minutes (1 day)

# =============================
# MACHINE NORMALIZER
# =============================
def normalize_machine(m):
    if not m or pd.isna(m):
        return None
    m = str(m).upper().strip()
    m = re.sub(r"[^A-Z0-9\-]", "", m)
    if m.startswith("MP") and "-" not in m:
        m = m.replace("MP", "MP-")
    return m

# =============================
# STEP 1: FILTER DAILY PLAN > 0
# =============================
valid = master[master["Daily Plan"] > 0].copy()

# =============================
# STEP 2: AGGREGATE AT CHILD PART LEVEL
# =============================
records = []

for child, grp in valid.groupby("Child Part", sort=False):

    demand_from_switches = (grp["Daily Plan"] * grp["Sub Count"]).sum()
    min_qty = grp["Minimum Quantity"].iloc[0]
    inventory = grp["Inventory_25"].iloc[0]

    machines = ",".join(grp["Vertical Machines"].astype(str))

    records.append({
        "Child Part": child,
        "Required Qty": min_qty + demand_from_switches,
        "Inventory_25": inventory,
        "Vertical Machines": machines
    })

agg = pd.DataFrame(records)
agg["Net Required Qty"] = agg["Required Qty"] - agg["Inventory_25"]

# =============================
# TRACKING
# =============================
machine_load = {m: 0.0 for m in ALLOWED_MACHINES}
machine_plan = defaultdict(list)

# =============================
# STEP 3: ASSIGN ALL QUANTITY (NO CAPACITY CHECK)
# =============================
for _, row in agg.iterrows():

    child = row["Child Part"]
    net_qty = row["Net Required Qty"]

    if net_qty <= 0:
        continue

    ct = cycle_time_min.get(child)
    if not ct or ct <= 0:
        continue

    total_time = net_qty * ct

    # Parse machines in given order
    raw = str(row["Vertical Machines"])
    tokens = re.split(r"[,\|/\\\n]+", raw)

    eligible = []
    for t in tokens:
        m = normalize_machine(t)
        if m and m in ALLOWED_MACHINES and m not in eligible:
            eligible.append(m)

    if not eligible:
        continue

    # 🔥 Assign EVERYTHING to first eligible machine
    m = eligible[0]

    machine_load[m] += total_time

    machine_plan[m].append({
        "Child Part": child,
        "Quantity": round(net_qty, 2),
        "Time Used (min)": round(total_time, 2)
    })

# =============================
# DISPLAY RESULTS
# =============================
print("\n========== MACHINE-WISE PLAN (1 DAY, NO CAPACITY LIMIT) ==========\n")
for m in ALLOWED_MACHINES:
    print(f"🔧 {m}")
    if machine_plan[m]:
        display(pd.DataFrame(machine_plan[m]))
    else:
        print("No allocation")
    print("-" * 60)

print("\n========== MACHINE LOAD SUMMARY (OVERLOAD VISIBLE) ==========\n")
display(pd.DataFrame([
    {
        "Machine": m,
        "Used (min)": round(machine_load[m], 2),
        "Capacity (min)": MACHINE_CAPACITY,
        "Overload (min)": round(machine_load[m] - MACHINE_CAPACITY, 2)
    }
    for m in ALLOWED_MACHINES
]))


========== MACHINE-WISE PLAN (1 DAY, NO CAPACITY LIMIT) ==========

🔧 MP-01


,Child Part,Quantity,Time Used (min)
0,S11434-002A0X,816.52,612.39
1,S12071-001A0X,41.35,24.12
2,S12094-003A0X,22.68,18.14
3,S41354-011A0X,2434.84,1826.13


------------------------------------------------------------
🔧 MP-05


,Child Part,Quantity,Time Used (min)
0,14SW030082-00001X0,689.71,517.28
1,14SW220197-00005X0,3399.84,1416.60
2,S12095-004A0X,1713.26,1284.94
3,S13083-003A0X,3283.61,1641.81
4,S13108-002A0X,18095.10,7690.42


------------------------------------------------------------
🔧 MP-10


,Child Part,Quantity,Time Used (min)
0,S11398-018A0X,187.97,140.98
1,S11398-030A0X,589.00,589.00
2,S13080-004A0X,97.52,37.38
3,S22166-016A0X,394.03,210.15
4,S33107-005A0X,108.29,54.15


------------------------------------------------------------
🔧 MP-17


,Child Part,Quantity,Time Used (min)
0,14SW110487-00009X0,285.26,114.10
1,14SW220201-00005X0,1152.10,691.26


------------------------------------------------------------

========== MACHINE LOAD SUMMARY (OVERLOAD VISIBLE) ==========



,Machine,Used (min),Capacity (min),Overload (min)
0,MP-01,2480.78,1320,1160.78
1,MP-05,12551.05,1320,11231.05
2,MP-10,1031.65,1320,-288.35
3,MP-17,805.36,1320,-514.64
